# `FFM_2_5_MLP_train` — landmark → z 的 MLP 训练

和 `FFM_2_4_Landmark2Z.ipynb` 分工：**2_4 出统计判据，2_5 只管训练。**
数据从 2_4 导出的 `landmark2z_cache.npz` 读，不重跑 encoder。

checkpoint 规矩完全照 `FFM_2_3_MultiDB.ipynb`：`best` / `latest` 双档，
存 `config` + `history` + `split` + 受试者名单 md5，支持中断续训，
test 集用 `RUN_TEST` 闸门锁住。

---

## 先把预期说清楚

`FFM_2_4` 的结论是 **CV-R² = 0.052 ± 0.015，事先约定的 0.1 停止规则已经触发**，
而且核方法选中的带宽退化到近似线性 —— 非线性一分钱没赚到。端到端 LSD 4.208 dB
对群体均值 4.253 dB，差 0.045 dB，**远低于 0.19 dB 的测量分辨率**，也就是
与均值基线不可区分。

所以这个 notebook 的目的**不是**指望 MLP 翻盘，而是：

> 论文里"我们也训了 MLP，没有超过 ridge"是审稿人想看到的一句话。
> 这一格把它变成一个有 checkpoint、有曲线、可复现的事实。

**判据是 ridge 的 CV-R² = 0.052。** MLP 打不过它就写进负结果，不要加容量。

## 关于 epoch 数

33 个训练被试、batch 8 → 每 epoch 约 **5 步**。`EPOCHS = 20` 就是 **100 步梯度**，
模型基本还没离开初始化，你会看到一条几乎平的曲线 —— 那不是拟合失败，是没开始训。
这个规模下跑 500 epoch 也就几十秒，没有理由省。默认给 20（你要的），
`CFG['EPOCHS']` 改成 500 再跑一次对比。

## 划分

| | 人数 | 来源 |
|---|---:|---|
| train | 33 | 41 个 AE-train 被试里切出 80% |
| val | 8 | 剩下的 20%，**只用来选 checkpoint** |
| test | 11 | AE-val + AE-test 的映射被试，**`RUN_TEST=True` 才碰** |

划分冻结到 `landmark2z_split.json`，键是受试者名单的 md5，名单变了会拒绝复用。

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 缓存
# ==========================================================================
import os, sys, json, time, hashlib
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
else:
    DRIVE = os.environ.get('ELEC5021_DRIVE', os.path.expanduser('~/Documents/26S2/ELEC5021'))

SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
CACHE    = f'{SAVE_DIR}/landmark2z_cache.npz'
assert os.path.exists(CACHE), (
    f'找不到 {CACHE} —— 先跑 FFM_2_4_Landmark2Z.ipynb 的最后一格（Cell 11 导出缓存）')

d = np.load(CACHE, allow_pickle=True)
L_sub, Z_sub = d['L_sub'], d['Z_sub']
cl, ae_part  = d['cl'], d['ae_part']
n_bins, latent_dim = int(d['n_bins']), int(d['latent_dim'])
SUBJ_HASH = str(d['subject_hash'])

print(f'缓存 <- {CACHE}')
print(f'  L_sub {L_sub.shape}  (feature={d["feature"]}, center={d["center"]})')
print(f'  Z_sub {Z_sub.shape}  ({n_bins} bins x {latent_dim})')
print(f'  {len(cl)} 人 | tier={list(d["tiers"])} | hash {SUBJ_HASH}')
print(f'  side 方差 {float(d["side_frac"]):.1%}，投影={bool(d["projected"])}')
from collections import Counter
print(f'  AE 分区: {dict(Counter(ae_part))}')

In [ ]:
# ==========================================================================
# Cell 1 — ★ 配置（唯一要动的地方）
# ==========================================================================
CFG = dict(
    tag          = 'L2Z_mlp_v1',
    K_VAR        = 0.95,        # 目标保留多少 latent 方差 -> 决定输出维度
    HIDDEN       = 128,
    N_LAYERS     = 2,
    DROPOUT      = 0.2,
    LR           = 1e-3,
    WEIGHT_DECAY = 1e-2,
    BATCH        = 8,
    EPOCHS       = 20,          # ★ 你要的 20。33 人 / batch 8 -> 每 epoch 5 步 = 100 步梯度
    VAL_FRAC     = 0.20,        # 从 41 个 AE-train 里切出多少做 val
    SEED         = 0,
    save_every   = 1,
    save_after   = 0,
    resume_lr_mult = 0.3,
)
RESUME = False                  # 改了 CFG 的结构项（HIDDEN/N_LAYERS/K_VAR）就必须 False

RIDGE_BASELINE = 0.0518         # FFM_2_4 的 ridge CV-R²，这是要打败的东西
STOP_RULE      = 0.10           # 事先约定的停止阈值

latest_path = f'{SAVE_DIR}/{CFG["tag"]}_latest.pth'
best_path   = f'{SAVE_DIR}/{CFG["tag"]}_best.pth'
split_path  = f'{SAVE_DIR}/landmark2z_split.json'

steps = int(np.ceil(41 * (1 - CFG['VAL_FRAC']) / CFG['BATCH']))
print(json.dumps(CFG, indent=1))
print(f'\n每 epoch 约 {steps} 步 -> {CFG["EPOCHS"]} epoch = 约 {steps*CFG["EPOCHS"]} 步梯度')
if CFG['EPOCHS'] * steps < 1000:
    print('⚠️ 少于 1000 步。这个规模下 500 epoch 也就几十秒 —— 先跑 20 看形状，')
    print('   然后把 EPOCHS 改成 500 再跑一次，两条曲线一起放进结果里。')

In [ ]:
# ==========================================================================
# Cell 2 — 划分（冻结到磁盘，键是受试者名单 md5）
#
# test = 11 个 AE 没见过的人，在 RUN_TEST 之前一律不碰。
# val  = 从 41 个 AE-train 里切 20%，只用来选 checkpoint。
# ==========================================================================
def build_split(path, hash_, ae_part, seed, val_frac):
    sp = json.load(open(path)) if os.path.exists(path) else {}
    if sp.get('hash') == hash_:
        print(f'  沿用已冻结划分 ({path})')
        return sp
    if sp:
        raise RuntimeError(
            f'受试者名单变了（存档 {sp.get("hash")} vs 现在 {hash_}）。'
            '重新划分会污染 test —— 先确认这是不是预期的，是的话手动删掉那个 json。')
    idx_tr_pool = np.where(ae_part == 'train')[0]
    rng = np.random.default_rng(seed)
    perm = rng.permutation(idx_tr_pool)
    n_va = int(round(val_frac * len(perm)))
    sp = {'hash': hash_, 'seed': seed, 'val_frac': val_frac,
          'train': perm[n_va:].tolist(), 'val': perm[:n_va].tolist(),
          'test': np.where(ae_part != 'train')[0].tolist()}
    json.dump(sp, open(path, 'w'), indent=1)
    print(f'  ★ 新划分，已写回 {path}')
    return sp

SP = build_split(split_path, SUBJ_HASH, ae_part, CFG['SEED'], CFG['VAL_FRAC'])
I_TR, I_VA, I_TE = (np.array(SP[k]) for k in ('train', 'val', 'test'))
S_TR, S_VA, S_TE = set(I_TR.tolist()), set(I_VA.tolist()), set(I_TE.tolist())
assert not (S_TR & S_VA), 'train/val 有重叠'
assert not ((S_TR | S_VA) & S_TE), '★ train/val 碰到了 test'
assert S_TR | S_VA | S_TE == set(range(len(cl))), '划分没有覆盖全部受试者'
print(f'train {len(I_TR)} | val {len(I_VA)} | test {len(I_TE)} 人')
print(f'  test 名单: {sorted(cl[I_TE])}')

# ---------------- 目标：latent 的主成分（统计量只从 train 折算） ----------------
mu_z = Z_sub[I_TR].mean(0)
Zc   = Z_sub - mu_z
U, S, Vt = np.linalg.svd(Zc[I_TR], full_matrices=False)
K = int(np.searchsorted(np.cumsum(S**2) / (S**2).sum(), CFG['K_VAR']) + 1)
P = Vt[:K]                                     # (K, 5952) 投影矩阵
Y = Zc @ P.T                                   # (52, K)
print(f'\n目标维度 K = {K}（{CFG["K_VAR"]:.0%} 方差，只用 train 的 {len(I_TR)} 人算）')

# ---------------- 特征标准化（同样只从 train 折算） ----------------
mu_x, sd_x = L_sub[I_TR].mean(0), L_sub[I_TR].std(0) + 1e-9
X = (L_sub - mu_x) / sd_x
print(f'X {X.shape} -> Y {Y.shape}')

def r2(pred, idx):
    """基线 = train 折的 Y 均值 —— 和 ridge 那边同一个口径"""
    ss_res = ((Y[idx] - pred) ** 2).sum()
    ss_tot = ((Y[idx] - Y[I_TR].mean(0)) ** 2).sum()
    return 1 - ss_res / ss_tot

In [ ]:
# ==========================================================================
# Cell 3 — 模型 · 优化器
# ==========================================================================
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(CFG['SEED']); np.random.seed(CFG['SEED'])
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


class LandmarkToLatent(nn.Module):
    """57 -> HIDDEN -> ... -> K。刻意做小：33 个 landmark、33 个训练被试，
    容量不是缺的那一块（ridge 已经证明连非线性都吃不到）。"""
    def __init__(self, d_in, d_out, hidden, n_layers, dropout):
        super().__init__()
        layers, d = [], d_in
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.GELU(), nn.Dropout(dropout)]
            d = hidden
        layers += [nn.Linear(d, d_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = LandmarkToLatent(X.shape[1], K, CFG['HIDDEN'], CFG['N_LAYERS'],
                         CFG['DROPOUT']).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['LR'],
                              weight_decay=CFG['WEIGHT_DECAY'])
criterion = nn.MSELoss()
print(f'参数 {sum(p.numel() for p in model.parameters()):,} '
      f'| 训练样本 {len(I_TR)} 人 —— 参数/样本 = '
      f'{sum(p.numel() for p in model.parameters())/len(I_TR):,.0f}')
print('⚠️ 参数远多于样本。这不是 bug，是这条线的处境本身。')

Xt = torch.tensor(X, dtype=torch.float32, device=device)
Yt = torch.tensor(Y, dtype=torch.float32, device=device)
train_loader = DataLoader(TensorDataset(Xt[I_TR], Yt[I_TR]),
                          batch_size=CFG['BATCH'], shuffle=True)

In [ ]:
# ==========================================================================
# Cell 4 — 训练（best / latest 双档，可中断续训）
# ==========================================================================
def save_ckpt(path, epoch, val_r2, history, with_opt):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if with_opt else None,
        'epoch': epoch, 'val_r2': val_r2, 'history': history,
        'config': CFG, 'split': SP, 'subject_hash': SUBJ_HASH,
        'K': K, 'P': P, 'mu_z': mu_z, 'mu_x': mu_x, 'sd_x': sd_x,
        'ridge_baseline': RIDGE_BASELINE,
    }, path)


start_epoch, best_val, history = 0, -np.inf, {'train_loss': [], 'val_r2': [], 'train_r2': []}
if RESUME and os.path.exists(latest_path):
    ck = torch.load(latest_path, map_location=device, weights_only=False)
    assert ck['subject_hash'] == SUBJ_HASH, '受试者名单变了，不能续训'
    assert ck['K'] == K, f"存档 K={ck['K']} vs 现在 {K} —— K_VAR 改过了，不能续训"
    model.load_state_dict(ck['model_state_dict'])
    if ck['optimizer_state_dict']:
        optimizer.load_state_dict(ck['optimizer_state_dict'])
    start_epoch, history = ck['epoch'] + 1, ck['history']
    best_val = max(history['val_r2']) if history['val_r2'] else -np.inf
    for g in optimizer.param_groups:
        g['lr'] = CFG['LR'] * CFG['resume_lr_mult']
    print(f"续训 <- epoch {start_epoch} | 最好 val R² {best_val:+.4f} | "
          f"lr -> {CFG['LR']*CFG['resume_lr_mult']:.2e}")
else:
    print('从零开始训练')

t0 = time.time()
for ep in range(start_epoch, CFG['EPOCHS']):
    model.train(); tot = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        tot += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        v = r2(model(Xt[I_VA]).cpu().numpy(), I_VA)
        t = r2(model(Xt[I_TR]).cpu().numpy(), I_TR)
    history['train_loss'].append(tot / len(I_TR))
    history['val_r2'].append(float(v)); history['train_r2'].append(float(t))

    if ep >= CFG['save_after']:
        if (ep + 1) % CFG['save_every'] == 0:
            save_ckpt(latest_path, ep, v, history, with_opt=True)
        if v > best_val:
            best_val = v
            save_ckpt(best_path, ep, v, history, with_opt=False)
    if ep % max(1, CFG['EPOCHS'] // 20) == 0 or ep == CFG['EPOCHS'] - 1:
        print(f'  ep {ep+1:>4}/{CFG["EPOCHS"]} | loss {tot/len(I_TR):.4f} '
              f'| train R² {t:+.4f} | val R² {v:+.4f}'
              + ('  ← best' if v == best_val else ''))

print(f'\n用时 {time.time()-t0:.1f}s | 最好 val R² = {best_val:+.4f} @ '
      f'epoch {int(np.argmax(history["val_r2"]))+1}')
print(f'ridge 基准 CV-R² = {RIDGE_BASELINE:+.4f}')
print(f'-> MLP {"打赢了" if best_val > RIDGE_BASELINE else "没打赢"} ridge'
      f'（注意 val 只有 {len(I_VA)} 人，单次估计噪声很大，不要当结论）')

In [ ]:
# ==========================================================================
# Cell 5 — 曲线
# ==========================================================================
import matplotlib.pyplot as plt

ep = np.arange(1, len(history['val_r2']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
ax[0].plot(ep, history['train_loss'], lw=1.5)
ax[0].set_title('train MSE'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, history['train_r2'], lw=1.5, label='train')
ax[1].plot(ep, history['val_r2'], lw=1.5, label='val')
ax[1].axhline(RIDGE_BASELINE, color='crimson', ls='--', lw=1.2,
              label=f'ridge CV-R² {RIDGE_BASELINE:.3f}')
ax[1].axhline(STOP_RULE, color='gray', ls=':', lw=1.2, label=f'停止规则 {STOP_RULE}')
ax[1].axhline(0, color='k', lw=.8)
ax[1].set_title('R²'); ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

gap = history['train_r2'][-1] - history['val_r2'][-1]
print(f'末轮 train−val R² 间隙 = {gap:+.4f}')
print('间隙大 = 在背 33 个人；间隙小且两条都贴着 0 = 根本没东西可学。'
      '这条线上后者才是预期。')

In [ ]:
# ==========================================================================
# Cell 6 — 载入 checkpoint（只做评估时从这里开始，跳过 Cell 4）
# ==========================================================================
ck = torch.load(best_path, map_location=device, weights_only=False)
assert ck['subject_hash'] == SUBJ_HASH, '受试者名单和存档不一致'
assert ck['K'] == K, f"存档 K={ck['K']} vs 现在 {K}"
model.load_state_dict(ck['model_state_dict']); model.eval()
history = ck['history']
print(f"载入 {os.path.basename(best_path)} | epoch {ck['epoch']+1} | "
      f"val R² {ck['val_r2']:+.4f}")
print(f"config: {json.dumps(ck['config'], ensure_ascii=False)}")

In [ ]:
# ==========================================================================
# Cell 7 — ★ TEST 评估（整个流程里只在最后跑一次）
#
# 规矩和 FFM_2_3 一样：调超参、选 epoch、比架构，全部看 val。
# test 只在结论定稿时跑一次，跑完不要再回去改任何东西。
# 这 11 个人里有 8 个是冻结的 CARLab test 受试者。
# ==========================================================================
RUN_TEST = False

if not RUN_TEST:
    print('RUN_TEST = False。要出最终数字时改成 True，且只改这一次。')
else:
    model.eval()
    with torch.no_grad():
        pred_te = model(Xt[I_TE]).cpu().numpy()
    r2_te = r2(pred_te, I_TE)
    print(f'{"MLP  test R²":22s}{r2_te:+.4f}   (n = {len(I_TE)} 人)')
    print(f'{"ridge CV-R² (参考)":22s}{RIDGE_BASELINE:+.4f}')
    print(f'{"停止规则":22s}{STOP_RULE:+.4f}')
    print()
    print('★ 报数时必须带上: n=11 人、33 个 landmark、目标是 latent 的前 '
          f'{K} 个主成分、DC–15.85 kHz。')
    print('★ 这个 R² 不能和重建的 1.2909 dB 或端到端的 4.208 dB 混在一张表里。')

    # 反投影回 latent，供 FFM_2_4 的 cell 10 口径做端到端 LSD
    Z_hat_te = pred_te @ P + mu_z
    np.savez(f'{SAVE_DIR}/{CFG["tag"]}_zhat_test.npz',
             cl=cl[I_TE], Z_hat=Z_hat_te, r2=r2_te)
    print(f'\nẑ 已存 -> {CFG["tag"]}_zhat_test.npz'
          f'（形状 {Z_hat_te.shape}，reshape 成 ({n_bins}, {latent_dim}) 喂 decoder）')

## 报数纪律

1. **判据是 ridge 的 CV-R² = 0.052，不是 0。** MLP 只有稳定超过它才值得写"有增益"，
   而 val 只有 8 人、test 只有 11 人，单次差异远在噪声里。

2. **`EPOCHS = 20` 的那条曲线不要单独放进论文。** 100 步梯度什么都说明不了。
   要么跑到 500 再报，要么两条一起放并说明。

3. **test 只跑一次。** 那 11 个人里有 8 个是冻结的 CARLab test 受试者，
   跑完再改上游就等于烧掉它。

4. **三个数不能同框**：重建 1.2909 dB / 端到端 4.208 dB / 这里的 R²。
   不同任务、不同口径。

## 已知的结论（跑之前就知道）

`FFM_2_4` 的结果：dCor²ᵤ = 0.071 (p = 0.006)，信号全在 6–16 kHz，
ridge CV-R² = 0.052 ± 0.015，端到端 4.208 dB vs 均值基线 4.253 dB
（差 0.045 dB，远低于 0.19 dB 的测量分辨率 —— 与均值不可区分）。

**这个 notebook 的产出是一个可复现的负结果，不是一次翻盘的尝试。**